# DataAgent — Full Pipeline Test

This notebook is a **developer-facing integration test** for the entire DataAgent stack.
Each section targets one specific feature or a combination of features. Run the cells top-to-bottom.
Phoenix tracing and CodeCarbon are intentionally excluded (not yet implemented).

**Excluded from scope:** Phoenix / OpenInference tracing, CodeCarbon emissions tracking.

---

## 0. Setup & Prerequisites

Adds the project root to `sys.path` and imports the core classes.
Configure the **provider**, **model**, and **API key** here — all subsequent sections inherit these values.

**Relevant code:** `Agent/data_agent.py` (`SalesDataAgent.__init__`), `Agent/config.py` (`AgentConfig`), `Agent/schema.py` (`DatabaseSchema`), `Agent/cache.py` (`RunCache`).

In [1]:
import sys, os

# Ensure the project root is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Configure these before running ───────────────────────────────────────────
PROVIDER       = "openai"          # "openai" or "ollama"
MODEL          = "gpt-4o-mini"     # e.g. "gpt-4o-mini", "llama3.2:3b"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # or set explicitly as a string
OLLAMA_URL     = "http://localhost:11434"       # ignored when provider=openai
DATA_PATH      = os.path.join(PROJECT_ROOT, "data", "Store_Sales_Price_Elasticity_Promotions_Data.parquet")
SCHEMA_YAML    = os.path.join(PROJECT_ROOT, "data", "sales_schema.yaml")
# ─────────────────────────────────────────────────────────────────────────────

from Agent.data_agent import SalesDataAgent
from Agent.config     import AgentConfig, StepConfig
from Agent.schema     import DatabaseSchema, TableSchema, ColumnSchema
from Agent.cache      import RunCache
import yaml, json, pprint

pp = pprint.PrettyPrinter(indent=2, width=120)


def agent_run(agent, *args, **kwargs):
    """Call agent.run() and normalize the return value to always be a dict.

    The legacy _run_with_evaluation() path (used when save_results=False and
    reuse_from=None) returns a (dict, score_variance) tuple. The new caching
    API path (save_results=True or reuse_from set) returns a plain dict.
    This helper unifies both so assertions work regardless of which path is taken.

    Relevant code: Agent/data_agent.py — SalesDataAgent.run(), _run_with_evaluation().
    """
    result = agent.run(*args, **kwargs)
    if isinstance(result, tuple):
        return result[0]
    return result


print("Setup complete.")
print(f"  Provider : {PROVIDER}")
print(f"  Model    : {MODEL}")
print(f"  Data     : {DATA_PATH}")

CodeCarbon is available
<module 'langgraph.version' from 'c:\\Users\\Davide\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\langgraph\\version.py'>
Setup complete.
  Provider : openai
  Model    : gpt-4o-mini
  Data     : c:\Users\Davide\Desktop\POLIMI\Tesi\GitHub_Code\DataAgent-1\data\Store_Sales_Price_Elasticity_Promotions_Data.parquet


---
## 1. Full Pipeline Run — Lookup → Analyze → Visualize

Runs the agent end-to-end through all three functional steps:
1. **`lookup_sales_data`** — LLM generates a DuckDB SQL query, executes it on the parquet file, returns a string table and a DataFrame.
2. **`analyzing_data`** — LLM interprets the query result and produces a short natural-language answer.
3. **`create_visualization`** — LLM infers a chart configuration (JSON) and generates matplotlib Python code.

Orchestration is managed by the **`decide_tool`** LangGraph node, which routes between steps.

**Relevant code:** `Agent/data_agent.py` — `SalesDataAgent.run_core()`, `SalesDataAgent._build_graph()`, `lookup_sales_data_core()`, `analyzing_data_core()`, `create_visualization_core()`, `decide_tool_core()`.

**Expected outcome:** `result["answer"]` has two entries (analysis text + matplotlib code), `result["sql_query"]` contains a valid SQL string, `result["data"]` is non-empty.

In [ ]:
agent_full = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    temperature=0.1,
)

result_full = agent_run(
    agent_full,
    "What is the total sales value per store for year 2021? Show a bar chart.",
    visualization_goal="Bar chart: total sales per store in 2021",
)

# ── Assertions ───────────────────────────────────────────────────────────────
assert isinstance(result_full, dict),               "result should be a dict"
assert "error" not in result_full,                  f"Unexpected error: {result_full.get('error')}"
assert result_full.get("sql_query"),                "sql_query should be set"
assert result_full.get("data"),                     "data (query result table) should be non-empty"
assert len(result_full.get("answer", [])) >= 1,     "answer list should contain at least the analysis text"

print("\n=== SQL Query ===")
print(result_full["sql_query"])
print("\n=== Analysis ===")
print(result_full["answer"][0] if result_full["answer"] else "(no answer)")
if len(result_full["answer"]) > 1:
    print("\n=== Visualization Code ===")
    print(result_full["answer"][1][:600], "...")
print("\n=== Chart Config ===")
pp.pprint(result_full.get("chart_config"))
print("\n[PASS] Full pipeline test")

Optionally execute the generated matplotlib code to render the chart inline.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

answers = result_full.get("answer", [])
if len(answers) > 1:
    chart_code = answers[-1]
    data_df    = result_full.get("data_df")
    config     = result_full.get("chart_config", {})
    exec_code  = chart_code.replace("plt.show()", "plt.show(); plt.close('all')")
    exec(exec_code, {"data_df": data_df, "config": config, "plt": plt, "pd": pd})
else:
    print("No visualization generated (check pipeline mode).")

---
## 2. Agent Modes — `lookup_only`, `no_vis` (analysis only), `full`

The agent supports three execution modes controlled by `lookup_only` and `no_vis` flags passed to `run()`.

| Mode | `lookup_only` | `no_vis` | Steps executed |
|------|:---:|:---:|---|
| Lookup only | `True` | — | `lookup_sales_data` |
| Analysis only | `False` | `True` | `lookup_sales_data` → `analyzing_data` |
| Full | `False` | `False` | `lookup_sales_data` → `analyzing_data` → `create_visualization` |

**Relevant code:** `Agent/data_agent.py` — `SalesDataAgent.run_core()` (branching logic at lines `~2274`, `~2296`, `~2325`).

In [ ]:
PROMPT_MODE = "What were the top 5 SKUs by total quantity sold in 2020?"

agent_mode = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    temperature=0.1,
)

# ── 2a: lookup_only ──────────────────────────────────────────────────────────
r_lookup = agent_run(agent_mode, PROMPT_MODE, lookup_only=True)
assert "error" not in r_lookup,            f"Error: {r_lookup.get('error')}"
assert r_lookup.get("data"),               "data should be non-empty in lookup_only mode"
assert not r_lookup.get("answer"),         "answer should be empty in lookup_only mode"
print("[PASS] lookup_only mode")
print("  SQL:", r_lookup["sql_query"][:120])

# ── 2b: no_vis (analysis only) ───────────────────────────────────────────────
r_novis = agent_run(agent_mode, PROMPT_MODE, no_vis=True)
assert "error" not in r_novis,             f"Error: {r_novis.get('error')}"
assert r_novis.get("data"),               "data should be non-empty in no_vis mode"
assert len(r_novis.get("answer", [])) == 1, "no_vis should produce exactly 1 answer (analysis)"
print("\n[PASS] no_vis (analysis-only) mode")
print("  Analysis:", r_novis["answer"][0][:200])

# ── 2c: full (default) ───────────────────────────────────────────────────────
r_full2 = agent_run(
    agent_mode,
    PROMPT_MODE,
    visualization_goal="Bar chart: top 5 SKUs by quantity sold",
)
assert "error" not in r_full2,             f"Error: {r_full2.get('error')}"
assert len(r_full2.get("answer", [])) >= 2, "full mode should produce analysis + chart code"
print("\n[PASS] full mode")

---
## 3. Multi-Table Schema

Tests the multi-table support by constructing a `DatabaseSchema` with two tables pointing to the same parquet file (simulating a scenario with a sales table and a promotions table).
When the number of tables exceeds `compact_threshold` (default 5), the agent runs a **two-step table selection**: first asks the LLM which tables are needed (using only names + descriptions), then passes full column details only for those tables.
Here we keep two tables (below the threshold) to verify the multi-table SQL join path.

**Relevant code:** `Agent/schema.py` — `DatabaseSchema`, `DatabaseSchema.should_use_table_selection()`, `DatabaseSchema.get_compact_summary()`, `DatabaseSchema.get_full_schema_str()`. `Agent/data_agent.py` — `lookup_sales_data_core()`, `select_relevant_tables()`.

In [ ]:
# Build a two-table schema that reuses the same parquet (sales + promo view)
multi_schema = DatabaseSchema(
    tables=[
        TableSchema(
            name="sales",
            description="Daily store-level sales transactions with pricing data",
            file_path=DATA_PATH,
            columns=[
                ColumnSchema("Store_Number",     "Unique store identifier",                    data_type="INTEGER"),
                ColumnSchema("SKU_Coded",        "Encoded product SKU",                        data_type="INTEGER"),
                ColumnSchema("Sold_Date",        "Transaction date (YYYY-MM-DD)",              data_type="VARCHAR",
                             example_values=["2019-01-01", "2021-11-15"]),
                ColumnSchema("Qty_Sold",         "Units sold",                                 data_type="INTEGER"),
                ColumnSchema("Total_Sale_Value", "Total revenue in USD",                       data_type="FLOAT"),
            ],
        ),
        TableSchema(
            name="promotions",
            description="Promotion flags per transaction (same grain as sales)",
            file_path=DATA_PATH,
            columns=[
                ColumnSchema("Store_Number",     "Unique store identifier (join key)",         data_type="INTEGER"),
                ColumnSchema("SKU_Coded",        "Encoded product SKU (join key)",             data_type="INTEGER"),
                ColumnSchema("Sold_Date",        "Transaction date (join key)",                data_type="VARCHAR"),
                ColumnSchema("On_Promo",         "Whether item was on promotion (1=yes, 0=no)",data_type="INTEGER",
                             example_values=["0", "1"]),
            ],
        ),
    ],
    compact_threshold=5,  # 2 tables < 5 → no table selection step needed here
)

print("Schema compact summary:")
print(multi_schema.get_compact_summary())
print(f"\nshould_use_table_selection: {multi_schema.should_use_table_selection()}")

agent_multi = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    schema=multi_schema,
    temperature=0.1,
)

r_multi = agent_run(
    agent_multi,
    "What is the average Total_Sale_Value for promoted vs non-promoted items in 2021?",
    no_vis=True,
)

assert "error" not in r_multi,  f"Error: {r_multi.get('error')}"
assert r_multi.get("data"),     "data should be non-empty"
print("\n=== SQL Query ===")
print(r_multi["sql_query"])
print("\n=== Analysis ===")
print(r_multi["answer"][0] if r_multi.get("answer") else "(no answer)")
print("\n[PASS] Multi-table schema test")

### 3b. Table Selection Step (threshold trigger)

Verify the two-step approach kicks in when tables exceed `compact_threshold`.
We set `compact_threshold=1` to force table selection even with two tables.

In [ ]:
multi_schema_large = DatabaseSchema(
    tables=multi_schema.tables,
    compact_threshold=1,   # Force selection step even with 2 tables
)
print(f"should_use_table_selection: {multi_schema_large.should_use_table_selection()}")
assert multi_schema_large.should_use_table_selection(), "Should trigger table selection with threshold=1"

agent_sel = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    schema=multi_schema_large,
    temperature=0.1,
)

r_sel = agent_run(
    agent_sel,
    "What is the total revenue per store in 2021?",
    lookup_only=True,
)
assert "error" not in r_sel, f"Error: {r_sel.get('error')}"
assert r_sel.get("data"),    "data should be non-empty"
print("SQL:", r_sel["sql_query"][:200])
print("\n[PASS] Table selection step triggered and executed")

---
## 4. Per-Step Best-of-N Sampling via `AgentConfig`

The new best-of-N mechanism is configured **per step** via `AgentConfig` / `StepConfig`,
rather than at the agent level. Each step independently samples `n` completions at linearly
spaced temperatures from `temp_min` to `temp_max`, scores them with `eval_fn` or `batch_eval_fn`,
and selects the best with `selection_fn`.

Here we configure `n=3` for `lookup_sales_data` (no eval function → argmax of zeros → first result)
and `n=2` for `analyzing_data` (no eval function → same).
The goal is to verify that the best-of-N execution loop runs, stores all N results, and returns one.

**Relevant code:** `Agent/config.py` — `StepConfig`, `AgentConfig`. `Agent/data_agent.py` — `SalesDataAgent._execute_step_with_config()`, `SalesDataAgent._create_llm()`.

In [7]:
from Agent.utils import make_csv_evaluator_no_gt, make_text_evaluator_no_gt

config_bon = AgentConfig(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
)

# Best-of-3 for lookup with consensus-based (no-GT) evaluator.
# NOTE: only batch_eval_fn is set (no eval_fn). During the N runs the per-run
# score prints "0.000" because eval_fn is None → that is expected. After all
# N runs complete, batch_eval_fn fires and computes pairwise IoU between the
# three DataFrames; those scores override the zeros and drive selection.
config_bon.lookup_sales_data = StepConfig(
    step_name="lookup_sales_data",
    n=3,
    temp_min=0.0,
    temp_max=0.5,
    batch_eval_fn=make_csv_evaluator_no_gt(),
    use_cache=False,
)

# Best-of-2 for analysis with no-GT LLM judge (eval_fn, scores each run individually).
config_bon.analyzing_data = StepConfig(
    step_name="analyzing_data",
    n=2,
    temp_min=0.1,
    temp_max=0.7,
    eval_fn=make_text_evaluator_no_gt(),
    use_cache=False,
)

agent_bon = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    agent_config=config_bon,
)

r_bon = agent_run(
    agent_bon,
    "What is the monthly total revenue for 2021?",
    no_vis=True,
)

assert "error" not in r_bon, f"Error: {r_bon.get('error')}"
assert r_bon.get("data"),    "data should be set"

# Verify that 3 lookup results were stored internally
lookup_results = agent_bon.current_run_step_results.get("lookup_sales_data", [])
assert len(lookup_results) == 3, f"Expected 3 lookup results, got {len(lookup_results)}"
print(f"Lookup N=3: {len(lookup_results)} result(s) stored")
print("  NOTE: per-run 'score=0.000' lines in the log above are expected —")
print("  batch_eval_fn replaces those zeros after all 3 runs complete.")

analyze_results = agent_bon.current_run_step_results.get("analyzing_data", [])
assert len(analyze_results) == 2, f"Expected 2 analysis results, got {len(analyze_results)}"
print(f"Analysis N=2: {len(analyze_results)} result(s) stored")

# Print the final batch scores stored on the best result (set by _execute_step_with_config)
if "_all_scores" in r_bon:
    print(f"\nFinal selection scores (batch IoU): {[f'{s:.3f}' for s in r_bon['_all_scores']]}")
    print(f"Best run index: {r_bon.get('_best_idx')}")

print("\n[PASS] Per-step Best-of-N test")

[Agent] Running best-of-1 with temperatures: [0.1]
Checking the model can run locally
OpenAI API is accessible
[Agent] Running agent without visualization
[lookup_sales_data] Running best-of-3 with temps ['0.00', '0.25', '0.50']
Generated SQL Query:
 SELECT DATE_TRUNC('month', CAST(Sold_Date AS DATE)) as Month, SUM(CAST(Total_Sale_Value AS DECIMAL)) as Total_Revenue 
FROM sales 
WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021%' 
GROUP BY Month 
ORDER BY Month;
  Run 1/3 (T=0.00): score=pending (batch eval at the end of the n run)
Generated SQL Query:
 SELECT DATE_TRUNC('month', CAST(Sold_Date AS DATE)) as Month, SUM(CAST(Total_Sale_Value AS DECIMAL)) as Total_Revenue 
FROM sales 
WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021%' 
GROUP BY Month 
ORDER BY Month;
  Run 2/3 (T=0.25): score=pending (batch eval at the end of the n run)
Generated SQL Query:
 SELECT DATE_TRUNC('month', CAST(Sold_Date AS DATE)) as Month, SUM(CAST(Total_Sale_Value AS DECIMAL)) as Total_Revenue 
FROM sales 
WHERE CAST

---
## 5. Caching — Save, Inspect, and Reuse

The `RunCache` stores all N results per step on disk (JSON) so they can be reused in subsequent runs
without re-invoking the LLM. Two cache modes exist:
- `cache_mode="skip"` — use the first cached result directly (fast replay).
- `cache_mode="auto"` — re-evaluate cached results with the current `eval_fn` and re-select.

This section:
1. Runs the agent with `save_results=True` and a fixed `run_id`.
2. Verifies the results are stored in `cache/agent_runs/<run_id>/`.
3. Runs again with `reuse_from=<run_id>` and verifies cache is hit (no LLM calls for cached steps).

**Relevant code:** `Agent/cache.py` — `RunCache.save_run()`, `RunCache.load_all_step_results()`, `RunCache.find_similar_runs()`, `RunCache.get_cache_stats()`. `Agent/data_agent.py` — `SalesDataAgent._execute_step_with_config()` (cache check), `SalesDataAgent._maybe_save_run_results()`.

In [2]:
import shutil

CACHE_DIR  = os.path.join(PROJECT_ROOT, "cache", "test_agent_runs")
CACHE_RUN_ID = "test-cache-section5"

# Clean up any previous test cache to start fresh
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)

agent_cache = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    cache_dir=CACHE_DIR,
)

# ── 5a: First run — save to cache ────────────────────────────────────────────
print("=== Run 1: saving to cache ===")
r_save = agent_cache.run(
    "What is the average Qty_Sold per Product_Class_Code for 2021?",
    no_vis=True,
    run_id=CACHE_RUN_ID,
    save_results=True,
)
assert "error" not in r_save, f"Error: {r_save.get('error')}"
print(f"Run ID: {r_save.get('run_id')}")

# Verify files on disk
cache = RunCache(CACHE_DIR)
stats = cache.get_cache_stats()
print(f"Cache stats after Run 1: {stats}")
assert stats["total_runs"] == 1, f"Expected 1 cached run, got {stats['total_runs']}"

meta = cache.load_run_metadata(CACHE_RUN_ID)
assert meta is not None,          "Metadata should be saved"
print(f"Cached prompt: '{meta['prompt']}'")

print("\n[PASS] Save to cache")

=== Run 1: saving to cache ===
Checking the model can run locally
OpenAI API is accessible
[Agent] Running agent without visualization
Generated SQL Query:
 SELECT Product_Class_Code, AVG(CAST(Qty_Sold AS INTEGER)) as Average_Qty_Sold 
FROM sales 
WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021%' 
GROUP BY Product_Class_Code 
ORDER BY Average_Qty_Sold DESC;

Agent response: The average Qty_Sold per Product_Class_Code for 2021 varies across different classes. The highest average is for Product_Class_Code 24425 at approximately 3.16, while the lowest is for Product_Class_Code 22825 at approximately 1.24.
[Agent] Run saved with ID: test-cache-section5
Run ID: test-cache-section5
Cache stats after Run 1: {'total_runs': 1, 'total_size_bytes': 9081, 'total_size_mb': 0.01, 'cache_dir': 'c:\\Users\\Davide\\Desktop\\POLIMI\\Tesi\\GitHub_Code\\DataAgent-1\\cache\\test_agent_runs'}
Cached prompt: 'What is the average Qty_Sold per Product_Class_Code for 2021?'

[PASS] Save to cache


In [3]:
# ── 5b: Second run — reuse from cache ────────────────────────────────────────
print("=== Run 2: reusing from cache ===")

# Set cache_mode="skip" on lookup and analyze so cached results are used directly
agent_cache.agent_config.lookup_sales_data.cache_mode = "skip"
agent_cache.agent_config.analyzing_data.cache_mode   = "skip"

r_reuse = agent_cache.run(
    "What is the average Qty_Sold per Product_Class_Code for 2021?",
    no_vis=True,
    reuse_from=CACHE_RUN_ID,
    save_results=True,
)
assert "error" not in r_reuse, f"Error: {r_reuse.get('error')}"

# The SQL and data should be identical to run 1 (same cache)
assert r_reuse.get("sql_query") == r_save.get("sql_query"), \
    "Reused SQL should match the original (cache hit)"
print(f"SQL match: {r_reuse['sql_query'][:80]}...")
print("\n[PASS] Reuse from cache")

=== Run 2: reusing from cache ===
[Agent] Loaded cached results from run: test-cache-section5
[Agent] Running agent without visualization
[lookup_sales_data] Found 1 cached result(s)
[lookup_sales_data] Using cached result (skip mode)
[analyzing_data] Found 1 cached result(s)
[analyzing_data] Using cached result (skip mode)

Agent response: The average Qty_Sold per Product_Class_Code for 2021 varies across different classes. The highest average is for Product_Class_Code 24425 at approximately 3.16, while the lowest is for Product_Class_Code 22825 at approximately 1.24.
[Agent] Run saved with ID: c030889b
SQL match: SELECT Product_Class_Code, AVG(CAST(Qty_Sold AS INTEGER)) as Average_Qty_Sold 
F...

[PASS] Reuse from cache


In [4]:
# ── 5c: Similar run discovery ─────────────────────────────────────────────────
print("=== Similar run discovery ===")
similar = cache.find_similar_runs("average quantity sold per product class 2021", top_k=5)
print(f"Similar runs found: {similar}")
assert len(similar) > 0, "Should find at least one similar run via Jaccard similarity"

# ── 5d: Exact match lookup ────────────────────────────────────────────────────
exact = cache.find_exact_match("What is the average Qty_Sold per Product_Class_Code for 2021?")
print(f"Exact match run ID: {exact}")
assert exact == CACHE_RUN_ID, "Exact match should return the original run_id"

# ── 5e: List cached runs ─────────────────────────────────────────────────────
runs = cache.list_runs()
print(f"Cached runs: {[r['run_id'] for r in runs]}")

print("\n[PASS] Cache inspection tests")

=== Similar run discovery ===
Similar runs found: ['test-cache-section5']
Exact match run ID: test-cache-section5
Cached runs: ['test-cache-section5']

[PASS] Cache inspection tests


---
## 6. Chain-of-Thought Iterative Refinement (`cot_n`)

When `cot_n > 1`, the agent feeds the previous iteration's output back into the prompt as a
"refinement block" (via `CoTRefinementLLM`). The loop stops early when the output similarity
exceeds `_COT_SIMILARITY_THRESHOLD` (0.95) or after `cot_n` iterations.

This test configures `cot_n=3` on the `lookup_sales_data` step and verifies that:
- The step runs (may stop early if output converges).
- The final SQL query is still valid and produces data.

**Relevant code:** `Agent/data_agent.py` — `CoTRefinementLLM`, `SalesDataAgent._apply_cot_iterations()`, `_COT_SIMILARITY_THRESHOLD`, `_extract_step_output()`.

In [3]:
from Agent.utils import make_csv_evaluator_no_gt, make_text_evaluator_no_gt

config_cot = AgentConfig(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
)

# Apply CoT refinement on lookup and analysis
config_cot.lookup_sales_data = StepConfig(
    step_name="lookup_sales_data",
    n=2,
    cot_n=3,
    temp_min=0.1,
    temp_max=0.3,
    use_cache=False,
)
config_cot.analyzing_data = StepConfig(
    step_name="analyzing_data",
    n=1,
    cot_n=2,
    temp_min=0.1,
    use_cache=False,
)

agent_cot = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    agent_config=config_cot,
)

r_cot = agent_run(
    agent_cot,
    "Find the top 3 stores with the highest total revenue in the second half of 2021 (Jul-Dec).",
    no_vis=True,
)

assert "error" not in r_cot, f"Error: {r_cot.get('error')}"
assert r_cot.get("data"),    "data should be non-empty after CoT refinement"

print("=== SQL Query after CoT refinement ===")
print(r_cot["sql_query"])
print("\n=== Analysis ===")
print(r_cot["answer"][0] if r_cot.get("answer") else "(no answer)")
print("\n[PASS] CoT iterative refinement test")

Checking the model can run locally
OpenAI API is accessible
[Agent] Running agent without visualization


Tool selected: lookup_sales_data
[lookup_sales_data] Running best-of-2 with temps ['0.10', '0.30']
[lookup_sales_data] CoT iteration 1/3: starting initial run...
Generated SQL Query:
 SELECT Store_Number, SUM(CAST(Total_Sale_Value AS FLOAT)) as Total_Revenue 
FROM sales 
WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021-07%' 
   OR CAST(Sold_Date AS VARCHAR) LIKE '%2021-08%' 
   OR CAST(Sold_Date AS VARCHAR) LIKE '%2021-09%' 
   OR CAST(Sold_Date AS VARCHAR) LIKE '%2021-10%' 
   OR CAST(Sold_Date AS VARCHAR) LIKE '%2021-11%' 
   OR CAST(Sold_Date AS VARCHAR) LIKE '%2021-12%' 
GROUP BY Store_Number 
ORDER BY Total_Revenue DESC 
LIMIT 3;

[lookup_sales_data] CoT iteration 2/3: starting refinement...
Generated SQL Query:
 SELECT Store_Number, SUM(CAST(Total_Sale_Value AS FLOAT)) as Total_Revenue 
FROM sales 
WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021-07%' 
   OR CAST(Sold_Date AS VARCHAR

---
## 7. Ground Truth (GT) Tracking

GT evaluation functions can be attached to each step's `gt_eval_fn`. They score the selected
result against a reference **but never influence selection** — they only log tracking metrics.
For `lookup_sales_data` the metric is **IoU** (row/column/table overlap between the GT CSV and
the query result). For `analyzing_data` it is a text similarity score (BLEU or LLM judge).

This test:
1. Runs a query whose expected result is known.
2. Attaches a GT CSV evaluator that scores the lookup result against that reference.
3. Verifies `_gt_score` appears in the result.

**Relevant code:** `Agent/utils.py` — `make_csv_evaluator()`, `make_text_evaluator()`, `make_vis_evaluator()`. `Agent/data_agent.py` — `SalesDataAgent._run_gt_eval()`, `SalesDataAgent._execute_step_with_config()`.

In [ ]:
import duckdb, pandas as pd, tempfile, os

# ── 7a: Generate ground truth CSV ─────────────────────────────────────────────
# Use DuckDB directly to create the reference answer for year 2022 (dataset range: 2021-11 → 2024-03)
GT_PROMPT = "What is the total revenue (Total_Sale_Value) per Store_Number for year 2022?"

con = duckdb.connect()
df_raw = pd.read_parquet(DATA_PATH)
con.register("sales", df_raw)
gt_df = con.execute(
    "SELECT Store_Number, SUM(Total_Sale_Value) AS total_revenue "
    "FROM sales WHERE CAST(Sold_Date AS VARCHAR) LIKE '2022%' "
    "GROUP BY Store_Number ORDER BY Store_Number"
).df()
con.close()

assert not gt_df.empty, "GT DataFrame should not be empty — check dataset date range"
print(f"GT rows: {len(gt_df)}")
print(gt_df.head())

GT_CSV_DIR  = tempfile.mkdtemp(prefix="gt_test_")
GT_CSV_PATH = os.path.join(GT_CSV_DIR, "gt_revenue_2022.csv")
gt_df.to_csv(GT_CSV_PATH, index=False)
print(f"\nGT CSV written to: {GT_CSV_PATH}")

# ── 7b: Attach GT evaluator ───────────────────────────────────────────────────
# gt_eval_fn requires a concrete reference answer. For lookup_sales_data the GT
# is the CSV computed above (objective, reproducible). For analyzing_data there
# is no fixed reference text, so no gt_eval_fn is set there — adding a fake one
# would be logically incorrect. Use eval_fn with metric='judge' instead if you
# need quality tracking on the analysis step without a GT text.
from Agent.utils import make_csv_evaluator

config_gt = AgentConfig(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
)

config_gt.lookup_sales_data = StepConfig(
    step_name="lookup_sales_data",
    n=1,
    use_cache=False,
    gt_eval_fn=make_csv_evaluator(GT_CSV_PATH),
)

agent_gt = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    agent_config=config_gt,
)

r_gt = agent_run(agent_gt, GT_PROMPT, no_vis=True)

assert "error" not in r_gt, f"Error: {r_gt.get('error')}"

lookup_result = agent_gt.current_run_step_results.get("lookup_sales_data", [{}])[0]
gt_score = lookup_result.get("_gt_score")

print(f"\nGT IoU score for lookup_sales_data step: {gt_score}")
assert gt_score is not None, "_gt_score should be set on lookup_sales_data result"
assert 0.0 <= gt_score <= 1.0, f"GT score out of range: {gt_score}"

print("\n=== SQL ===")
print(r_gt["sql_query"])
print("\n[PASS] Ground truth tracking test")

### 7c. Full GT Evaluation from `benchmark_dataset.json`

Runs all 3 benchmark prompts through the full pipeline attaching GT evaluators to **every step** that has a ground truth:
- `lookup_sales_data` → `make_csv_evaluator` (row-level IoU vs GT CSV)
- `analyzing_data` → `make_text_evaluator` with BLEU vs `gt_analysis`
- `create_visualization` → `make_vis_evaluator` with LLM judge vs `gt_chart_config` + `gt_chart_code` (skipped when `gt_chart_config` is null)

All `_gt_score` values are collected and printed in a summary table at the end.

**Relevant code:** `Agent/utils.py` — `make_csv_evaluator()`, `make_text_evaluator()`, `make_vis_evaluator()`, `judge_visualization()`. `Agent/data_agent.py` — `SalesDataAgent._run_gt_eval()`.

In [ ]:
import json, tempfile, os
from Agent.utils import make_csv_evaluator, make_text_evaluator, make_vis_evaluator

BENCHMARK_PATH = os.path.join(PROJECT_ROOT, "evaluation", "benchmark_dataset.json")
with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

print(f"Loaded {len(benchmark)} benchmark entries\n")

summary = []  # collect per-entry scores for the final table

for i, entry in enumerate(benchmark):
    prompt           = entry["prompt"]
    vis_goal         = entry.get("visualization_goal")
    gt_data_csv      = entry["gt_data"]          # CSV string
    gt_analysis      = entry["gt_analysis"]
    gt_chart_config  = entry.get("gt_chart_config")   # None for entry 3
    gt_chart_code    = entry.get("gt_chart_code")
    explicit_req     = entry.get("explicit_requirements")
    has_vis          = gt_chart_config is not None

    print(f"{'='*60}")
    print(f"Entry {i+1}/{len(benchmark)}: {prompt[:80]}")
    print(f"  Visualization: {'yes' if has_vis else 'no'}")

    # ── Write GT data CSV to a temp file ──────────────────────────────────────
    gt_csv_dir  = tempfile.mkdtemp(prefix=f"gt_bench_{i}_")
    gt_csv_path = os.path.join(gt_csv_dir, "gt_data.csv")
    with open(gt_csv_path, "w") as f:
        f.write(gt_data_csv)

    # ── Build AgentConfig with GT evaluators ──────────────────────────────────
    cfg = AgentConfig(model=MODEL, provider=PROVIDER, openai_api_key=OPENAI_API_KEY)

    cfg.lookup_sales_data = StepConfig(
        step_name="lookup_sales_data",
        n=1,
        use_cache=False,
        gt_eval_fn=make_csv_evaluator(gt_csv_path),
    )

    cfg.analyzing_data = StepConfig(
        step_name="analyzing_data",
        n=1,
        use_cache=False,
        gt_eval_fn=make_text_evaluator(
            ground_truth_text=gt_analysis,
            metric="bleu",
        ),
    )

    if has_vis:
        cfg.create_visualization = StepConfig(
            step_name="create_visualization",
            n=1,
            use_cache=False,
            gt_eval_fn=make_vis_evaluator(
                ground_truth_config=gt_chart_config,
                ground_truth_code=gt_chart_code,
                explicit_requirements=explicit_req,
                judge_model=MODEL,
                provider=PROVIDER,
                openai_api_key=OPENAI_API_KEY,
            ),
        )

    # ── Run agent ─────────────────────────────────────────────────────────────
    agent_bench = SalesDataAgent(
        model=MODEL,
        provider=PROVIDER,
        openai_api_key=OPENAI_API_KEY,
        ollama_url=OLLAMA_URL,
        agent_config=cfg,
    )

    run_kwargs = dict(no_vis=not has_vis)
    if has_vis and vis_goal:
        run_kwargs["visualization_goal"] = vis_goal

    result = agent_run(agent_bench, prompt, **run_kwargs)

    # ── Collect GT scores ─────────────────────────────────────────────────────
    lookup_res  = agent_bench.current_run_step_results.get("lookup_sales_data",  [{}])[0]
    analyze_res = agent_bench.current_run_step_results.get("analyzing_data",     [{}])[0]
    vis_res     = agent_bench.current_run_step_results.get("create_visualization",[{}])[0]

    score_lookup  = lookup_res.get("_gt_score")
    score_analyze = analyze_res.get("_gt_score")
    score_vis     = vis_res.get("_gt_score") if has_vis else None

    print(f"  GT scores → lookup: {score_lookup:.3f}  |  "
          f"analysis (BLEU): {score_analyze:.3f}  |  "
          f"vis: {f'{score_vis:.3f}' if score_vis is not None else 'n/a'}")

    summary.append({
        "entry": i + 1,
        "prompt": prompt[:55] + "..." if len(prompt) > 55 else prompt,
        "lookup_iou":   score_lookup,
        "analysis_bleu": score_analyze,
        "vis_judge":    score_vis,
    })

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"{'#':<4} {'Prompt':<57} {'Lookup IoU':>10} {'BLEU':>6} {'Vis':>6}")
print("-" * 88)
for row in summary:
    vis_str = f"{row['vis_judge']:.3f}" if row["vis_judge"] is not None else "  n/a"
    print(f"{row['entry']:<4} {row['prompt']:<57} "
          f"{row['lookup_iou']:>10.3f} {row['analysis_bleu']:>6.3f} {vis_str:>6}")

print("\n[PASS] Full benchmark GT evaluation")

---
## 8. YAML-Based Configuration (`run_config.yaml` + `AgentConfig.from_yaml`)

The unified YAML config (`config/run_config.yaml`) is the preferred way to configure complex runs.
`AgentConfig.from_yaml()` parses it and returns `(AgentConfig, run_params, DatabaseSchema)`.
Then `run_agent.py` assembles these into a `SalesDataAgent` and calls `run()`.

This section:
1. Loads the existing `config/run_config.yaml` via `AgentConfig.from_yaml()` and inspects the result.
2. Invokes `run_agent.py` as a subprocess to verify the CLI entry point works end-to-end.

**Relevant code:** `Agent/config.py` — `AgentConfig.from_yaml()`. `run_agent.py` — `main()`.

In [ ]:
# ── 8a: Python API ────────────────────────────────────────────────────────────
YAML_PATH = os.path.join(PROJECT_ROOT, "config", "run_config.yaml")

agent_cfg, run_params, schema_from_yaml = AgentConfig.from_yaml(YAML_PATH)

print("=== AgentConfig from YAML ===")
pp.pprint(agent_cfg.to_dict())

print("\n=== run_params ===")
safe_params = {k: v for k, v in run_params.items() if k != "tracing"}
pp.pprint(safe_params)

assert agent_cfg.model,                 "model should be set"
assert agent_cfg.provider,              "provider should be set"
assert run_params.get("prompt"),        "prompt should be set in run_config.yaml"

print("\n=== Schema from YAML ===")
if schema_from_yaml:
    for t in schema_from_yaml.tables:
        print(f"  Table: {t.name} ({len(t.columns)} columns) → {t.file_path}")
else:
    print("  No schema defined in YAML (single-table default mode)")

print("\n[PASS] AgentConfig.from_yaml() parsing test")

In [ ]:
# ── 8b: Subprocess call to run_agent.py ──────────────────────────────────────
import subprocess

proc = subprocess.run(
    [sys.executable, "run_agent.py", YAML_PATH],
    capture_output=True,
    text=True,
    cwd=PROJECT_ROOT,
    timeout=120,
)
print("STDOUT (first 2000 chars):")
print(proc.stdout[:2000])
if proc.returncode != 0:
    print("STDERR:")
    print(proc.stderr[:1000])

assert proc.returncode == 0, f"run_agent.py exited with code {proc.returncode}\n{proc.stderr[:500]}"
print("\n[PASS] run_agent.py CLI test")

---
## 9. Flask API (`agent_api.py`)

The Flask application (`agent_api.py`) exposes three endpoints:
- `GET /health` — returns `{"status": "healthy", "model": "..."}`
- `GET /models` — lists available Ollama models (fails gracefully when Ollama is not running)
- `POST /call-agent` — runs the agent with a JSON payload

This section starts the server in a background thread (using the Flask test client, not an actual
network socket) to avoid port conflicts and keep the test self-contained.

**Relevant code:** `agent_api.py` — `call_agent()`, `health()`, `list_models()`.

In [ ]:
# Import the Flask app directly and use the test client (no network socket needed)
import importlib, sys

# Remove cached module to allow fresh import with correct env
if "agent_api" in sys.modules:
    del sys.modules["agent_api"]

sys.path.insert(0, PROJECT_ROOT)
import agent_api as api_module

flask_app = api_module.app
flask_app.config["TESTING"] = True
client = flask_app.test_client()

# ── 9a: /health ───────────────────────────────────────────────────────────────
resp_health = client.get("/health")
print(f"/health status code: {resp_health.status_code}")
health_data = resp_health.get_json()
print(f"/health response: {health_data}")
# Accept both healthy and unhealthy (depends on provider availability in CI)
assert resp_health.status_code in (200, 503), f"Unexpected status code: {resp_health.status_code}"
print("[PASS] /health endpoint")

In [ ]:
# ── 9b: /models ───────────────────────────────────────────────────────────────
resp_models = client.get("/models")
print(f"/models status code: {resp_models.status_code}")
models_data = resp_models.get_json()
print(f"/models response: {models_data}")
# This can fail with 500 if Ollama is not running — both outcomes are valid
assert resp_models.status_code in (200, 500), f"Unexpected status: {resp_models.status_code}"
print("[PASS] /models endpoint")

In [ ]:
# ── 9c: /call-agent — missing prompt (400) ───────────────────────────────────
resp_no_prompt = client.post("/call-agent", json={})
assert resp_no_prompt.status_code == 400, "Missing prompt should return 400"
print(f"[PASS] /call-agent with missing prompt returns 400: {resp_no_prompt.get_json()}")

In [ ]:
# ── 9d: /call-agent — valid prompt ───────────────────────────────────────────
payload = {
    "prompt": "What is the total revenue for 2021?",
    "model": MODEL,
    "lookup_only": True,
}
if PROVIDER == "openai":
    payload["model"] = MODEL

resp_agent = client.post("/call-agent", json=payload)
print(f"/call-agent status code: {resp_agent.status_code}")
agent_data = resp_agent.get_json()

if resp_agent.status_code == 200:
    print(f"status : {agent_data.get('status')}")
    print(f"sql_query: {str(agent_data.get('sql_query', ''))[:120]}")
    assert agent_data.get("status") == "success", f"Expected success: {agent_data}"
    print("[PASS] /call-agent valid prompt")
else:
    print(f"[WARN] /call-agent returned {resp_agent.status_code}: {agent_data}")
    print("(May fail if OpenAI key is not set or model is not reachable in this environment)")

---
## 10. `step_overrides` — Runtime Per-Step Config Overrides

`run()` accepts a `step_overrides` dict that temporarily patches specific step parameters for that
single call without permanently modifying the `AgentConfig`. This is useful for ad-hoc experiments.

This test bumps `analyzing_data` to `n=2` and `temp_max=0.8` via `step_overrides` and verifies
the override is applied (two analysis results stored) and then rolled back.

**Relevant code:** `Agent/data_agent.py` — `SalesDataAgent.run()` (`step_overrides` block, ~line 2505).

In [ ]:
config_base = AgentConfig(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
)
# Default: n=1 for all steps
config_base.analyzing_data.n = 1

agent_ovr = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    agent_config=config_base,
)

# Override at call time: bump analyzing_data to n=2
r_ovr = agent_ovr.run(
    "How many unique SKUs were sold in 2020?",
    no_vis=True,
    save_results=True,      # triggers new-API path which applies overrides
    step_overrides={
        "analyzing_data": {"n": 2, "temp_max": 0.8, "use_cache": False},
        "lookup_sales_data": {"use_cache": False},
    },
)

assert "error" not in r_ovr, f"Error: {r_ovr.get('error')}"

analyze_results = agent_ovr.current_run_step_results.get("analyzing_data", [])
print(f"analyzing_data results count: {len(analyze_results)}")
assert len(analyze_results) == 2, f"Expected 2 (from override), got {len(analyze_results)}"

# Verify the base config was NOT permanently modified
assert agent_ovr.agent_config.analyzing_data.n == 1, "Base config n should still be 1 after override"
print("Base config.analyzing_data.n restored to:", agent_ovr.agent_config.analyzing_data.n)
print("\n[PASS] step_overrides test")

---
## 11. `StepConfig` — Disabled Step

When `StepConfig.enabled = False`, the middleware in `_execute_step_with_config()` skips the step
entirely and returns the state unchanged. This can be used to skip visualization or analysis
without touching the routing flags.

This test disables `create_visualization` and verifies no chart code is generated.

**Relevant code:** `Agent/data_agent.py` — `SalesDataAgent._execute_step_with_config()` (enabled check).

In [ ]:
config_disabled = AgentConfig(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
)
config_disabled.create_visualization.enabled = False

agent_dis = SalesDataAgent(
    model=MODEL,
    provider=PROVIDER,
    openai_api_key=OPENAI_API_KEY,
    ollama_url=OLLAMA_URL,
    agent_config=config_disabled,
)

r_dis = agent_run(
    agent_dis,
    "Show total sales per year. Create a line chart.",
    visualization_goal="Line chart: total sales per year",
)

assert "error" not in r_dis, f"Error: {r_dis.get('error')}"

# Visualization step was disabled; chart_config and chart code should not be generated
vis_results = agent_dis.current_run_step_results.get("create_visualization", [])
if vis_results:
    print(f"WARNING: vis step ran despite being disabled ({len(vis_results)} result(s))")
else:
    print("create_visualization was correctly skipped (no results stored)")

print(f"answer entries: {len(r_dis.get('answer', []))}")
print("\n[PASS] Disabled step test")

---
## 12. Schema API — `DatabaseSchema` Methods

Directly tests the `DatabaseSchema` helper methods used internally by the agent:
- `get_compact_summary()` — table names + descriptions only (used for table selection prompt)
- `get_full_schema_str()` — full column details (used for SQL generation)
- `get_full_schema_str(table_names=[...])` — filtered by table subset
- `should_use_table_selection()` — threshold check

**Relevant code:** `Agent/schema.py` — `DatabaseSchema`.

In [ ]:
test_schema = DatabaseSchema(
    tables=[
        TableSchema("sales", "Sales fact table", DATA_PATH, [
            ColumnSchema("Store_Number", "Store ID", "INTEGER"),
            ColumnSchema("Sold_Date",    "Transaction date", "VARCHAR", example_values=["2019-01-01"]),
            ColumnSchema("Total_Sale_Value", "Revenue USD", "FLOAT"),
        ]),
        TableSchema("products", "Product dimension", DATA_PATH, [
            ColumnSchema("SKU_Coded",        "Product SKU", "INTEGER"),
            ColumnSchema("Product_Class_Code", "Category", "INTEGER"),
        ]),
        TableSchema("stores", "Store dimension", DATA_PATH, [
            ColumnSchema("Store_Number", "Store ID", "INTEGER"),
        ]),
    ],
    compact_threshold=2,   # 3 tables > 2 → should trigger selection
)

# compact summary
compact = test_schema.get_compact_summary()
assert "sales" in compact and "products" in compact and "stores" in compact
print("=== Compact Summary ===")
print(compact)

# full schema string
full = test_schema.get_full_schema_str()
assert "Store_Number" in full and "example_values" not in full  # examples embedded differently
print("\n=== Full Schema (all tables) ===")
print(full[:600])

# filtered schema
filtered = test_schema.get_full_schema_str(table_names=["sales"])
assert "sales" in filtered
assert "products" not in filtered
print("\n=== Filtered Schema (sales only) ===")
print(filtered)

# threshold check
assert test_schema.should_use_table_selection() is True
print("\nshould_use_table_selection: True  ✓")

# get_table
t = test_schema.get_table("products")
assert t is not None and t.name == "products"
print("\n[PASS] DatabaseSchema API test")

---
## 13. RunCache — Full CRUD

Directly exercises the `RunCache` API: save, load metadata, load step results, delete, clear, stats.

**Relevant code:** `Agent/cache.py` — `RunCache.save_run()`, `RunCache.load_run_metadata()`,
`RunCache.load_step_results()`, `RunCache.delete_run()`, `RunCache.clear_cache()`, `RunCache.get_cache_stats()`.

In [ ]:
import shutil, uuid

CRUD_CACHE_DIR = os.path.join(PROJECT_ROOT, "cache", "test_crud")
if os.path.exists(CRUD_CACHE_DIR):
    shutil.rmtree(CRUD_CACHE_DIR)

crud_cache = RunCache(CRUD_CACHE_DIR)
run_id_a = "crud-run-A"
run_id_b = "crud-run-B"

# ── Save ─────────────────────────────────────────────────────────────────────
crud_cache.save_run(
    run_id=run_id_a,
    prompt="How many rows in 2021?",
    agent_config={"model": MODEL, "provider": PROVIDER},
    step_results={"lookup_sales_data": [{"sql_query": "SELECT COUNT(*) FROM sales WHERE ...", "data": "42"}]},
    final_result={"answer": ["There are 42 rows."]},
)
crud_cache.save_run(
    run_id=run_id_b,
    prompt="What is total revenue 2021?",
    agent_config={"model": MODEL, "provider": PROVIDER},
    step_results={"lookup_sales_data": [{"sql_query": "SELECT SUM(...) FROM sales ...", "data": "999"}]},
    final_result={"answer": ["Total revenue is $999."]},
)
assert crud_cache.get_cache_stats()["total_runs"] == 2
print("Saved 2 runs.")

# ── Load metadata ─────────────────────────────────────────────────────────────
meta_a = crud_cache.load_run_metadata(run_id_a)
assert meta_a["prompt"] == "How many rows in 2021?"
print(f"Loaded metadata for {run_id_a}: prompt='{meta_a['prompt']}'")

# ── Load step results ─────────────────────────────────────────────────────────
step_results_a = crud_cache.load_step_results(run_id_a, "lookup_sales_data")
assert step_results_a and step_results_a[0]["data"] == "42"
print(f"Loaded step results for {run_id_a}: {step_results_a}")

# ── Delete one run ────────────────────────────────────────────────────────────
deleted = crud_cache.delete_run(run_id_a)
assert deleted is True
assert crud_cache.get_cache_stats()["total_runs"] == 1
print(f"Deleted {run_id_a}. Remaining runs: 1")

# ── Clear all ─────────────────────────────────────────────────────────────────
n_deleted = crud_cache.clear_cache()
assert n_deleted == 1
assert crud_cache.get_cache_stats()["total_runs"] == 0
print(f"Cleared cache ({n_deleted} run deleted). Remaining runs: 0")

print("\n[PASS] RunCache CRUD test")

---
## Summary

| Section | Feature tested | Key files |
|---------|---------------|-----------|
| 1 | Full pipeline (lookup → analyze → visualize) | `data_agent.py` |
| 2 | Agent modes (`lookup_only`, `no_vis`, `full`) | `data_agent.py` |
| 3 | Multi-table schema + table selection | `schema.py`, `data_agent.py` |
| 4 | Per-step best-of-N via `AgentConfig` | `config.py`, `data_agent.py`, `utils.py` |
| 5 | Caching: save / reuse / inspect | `cache.py`, `data_agent.py` |
| 6 | CoT iterative refinement (`cot_n`) | `data_agent.py` (`CoTRefinementLLM`) |
| 7 | Ground truth tracking (`gt_eval_fn`) | `utils.py`, `data_agent.py` |
| 8 | YAML config + CLI (`run_agent.py`) | `config.py`, `run_agent.py` |
| 9 | Flask API endpoints | `agent_api.py` |
| 10 | `step_overrides` (runtime config patches) | `data_agent.py` |
| 11 | Disabled step (`StepConfig.enabled=False`) | `data_agent.py` |
| 12 | `DatabaseSchema` API methods | `schema.py` |
| 13 | `RunCache` CRUD | `cache.py` |